# R2 Storage Test and Bucket Inspector

Part 1 exercises the `storage` package public API (round-trip upload, verify, download, list, prefix filter) to confirm the gcs/r2 to storage refactor works.

Part 2 keeps low-level bucket inspection and maintenance (object sizes and timestamps, prefix deletion, raw byte put/get). These operations are not exposed by the `storage` package, so they use the same client the package uses via `_r2_client()`.

## Part 1: `storage` package

In [ ]:
import os

from dotenv import load_dotenv
from shared.repo import REPO_ROOT
from storage import(
    delete_from_r2,
    download_from_r2,
    fetch_uploaded_r2_keys,
    r2_key_exists,
    r2_object_md5,
    upload_to_r2,
    verify_upload,
)
from storage.transfer import _local_md5_b64

load_dotenv(dotenv_path=REPO_ROOT / ".env")

print(f"Endpoint : {os.environ['ENDPOINT_URL']}")
print(f"Bucket   : {os.environ['BUCKET']}")

### Upload, verify, download round-trip

In [ ]:
TEST_KEY = "r2_inspect_test/roundtrip.txt"
local_src = REPO_ROOT / "tmp" / "r2_roundtrip_src.txt"
local_dst = REPO_ROOT / "tmp" / "r2_roundtrip_dst.txt"
local_src.parent.mkdir(parents=True, exist_ok=True)
local_src.write_text("hello from r2_inspect.ipynb")

upload_md5 = _local_md5_b64(local_src)
upload_to_r2(local_src, TEST_KEY, extra_metadata={"gcs-md5": upload_md5})
print(f"exists after upload : {r2_key_exists(TEST_KEY)}")
print(f"verify_upload       : {verify_upload(TEST_KEY)}")
print(f"upload md5          : {upload_md5}")
print(f"r2 stored md5       : {r2_object_md5(TEST_KEY)}")

# Re-hash the downloaded bytes and compare against the metadata md5 stored at upload.
try:
    download_from_r2(TEST_KEY, local_dst, verify_md5=True)
    print(f"download md5        : {_local_md5_b64(local_dst)}")
    print(f"integrity verified  : True")
except Exception as e:
    print(f"integrity verified  : False")
    print(f"error               : {e}")

delete_from_r2(TEST_KEY)
print(f"exists after delete : {r2_key_exists(TEST_KEY)}")

### List keys

In [ ]:
keys = fetch_uploaded_r2_keys()
print(f"{len(keys)} object(s) in bucket")
head = 20
print(f"first {head} keys:")
for k in sorted(keys)[:head]:
    print(f"  {k}")

### Filter by prefix

In [ ]:
PREFIX = "arc-institute-virtual-cell-atlas/scbasecount/2026-01-12/h5ad/GeneFull/Homo_sapiens/"
filtered_keys = fetch_uploaded_r2_keys(PREFIX)
print(f"{len(filtered_keys)} object(s) under '{PREFIX}'")
for k in sorted(filtered_keys):
    print(f"  {k}")

## Part 2: Bucket inspection and maintenance

The cells below use the shared client from `storage.r2._r2_client()` for operations the package does not expose.

In [ ]:
from storage.r2 import _r2_client, delete_from_r2, delete_r2_prefix

BUCKET = os.environ["BUCKET"]
client = _r2_client()

### List all objects with sizes, dump to cache

In [ ]:
CACHE_DIR = REPO_ROOT / "output/r2"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
cache_file = CACHE_DIR / "r2_files_cache.txt"

cache_prefetch = cache_file.read_text() if cache_file.exists() else ""

paginator = client.get_paginator("list_objects_v2")
objects = []
for page in paginator.paginate(Bucket=BUCKET):
    objects.extend(page.get("Contents") or [])

if not objects:
    print("Bucket is empty.")
    cache_file.write_text("")
else:
    lines = []
    for obj in objects:
        size_mb = obj["Size"] / 1024 / 1024
        uploaded = obj["LastModified"].strftime("%Y-%m-%d %H:%M")
        lines.append(f"  {obj['Key']}  ({size_mb:.1f} MB)  {uploaded}\n")
    cache_file.write_text("".join(lines))
    print(f"{len(objects)} object(s) written to {cache_file}")

cache_postfetch = cache_file.read_text()
print("Cache is clean." if cache_prefetch == cache_postfetch else "Cache changed since last run. Please check the file.")

### Show unique prefixes

In [ ]:
prefix_counts = {}
for obj in objects:
    key = obj["Key"]
    prefix = key.split("/", 1)[0] if "/" in key else key
    prefix_counts[prefix] = prefix_counts.get(prefix, 0) + 1

if not prefix_counts:
    print("No prefixes found.")
else:
    print(f"Unique prefixes ({len(prefix_counts)}):")
    for p in sorted(prefix_counts):
        print(f"- {p}: {prefix_counts[p]} file(s)")

### Filter by prefix (with sizes)

In [ ]:
PREFIX = "arc-institute-virtual-cell-atlas/scbasecount/2026-01-12/h5ad/GeneFull/Homo_sapiens/"

paginator = client.get_paginator("list_objects_v2")
filtered = []
for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    filtered.extend(page.get("Contents") or [])

if not filtered:
    print(f"No objects found under '{PREFIX}'.")
else:
    print(f"{len(filtered)} object(s) under '{PREFIX}':")
    for obj in filtered:
        size_mb = obj["Size"] / 1024 / 1024
        print(f"  {obj['Key']}  ({size_mb:.1f} MB)")

### Delete files

In [ ]:
# delete_from_r2("test.txt")

### Delete files with a given prefix

In [ ]:
DELETE_PREFIX = "r2_inspect_test/"

paginator = client.get_paginator("list_objects_v2")
to_delete = []
for page in paginator.paginate(Bucket=BUCKET, Prefix=DELETE_PREFIX):
    to_delete.extend(page.get("Contents") or [])

if not to_delete:
    print(f"No objects found under '{DELETE_PREFIX}'. Nothing to delete.")
else:
    print(f"Found {len(to_delete)} object(s) under '{DELETE_PREFIX}':")
    total_mb = 0.0
    for obj in to_delete:
        size_mb = obj["Size"] / 1024 / 1024
        total_mb += size_mb
        print(f"  {obj['Key']}  ({size_mb:.1f} MB)")
    print(f"\nTotal: {total_mb:.1f} MB")

    confirm = input(f"\nType 'yes' to delete all {len(to_delete)} objects: ")
    if confirm.strip().lower() != "yes":
        print("Aborted.")
    else:
        deleted = delete_r2_prefix(DELETE_PREFIX)
        print(f"Deleted {len(deleted)} object(s) under '{DELETE_PREFIX}'.")

### Raw byte upload and read back

In [ ]:
KEY = "r2_inspect_test/rawbyte.txt"
body = b"hello from r2_inspect.ipynb"

client.put_object(Bucket=BUCKET, Key=KEY, Body=body)
print(f"Uploaded '{KEY}' ({len(body)} bytes) to r2://{BUCKET}/{KEY}")

response = client.get_object(Bucket=BUCKET, Key=KEY)
content = response["Body"].read().decode()
print(f"Contents of '{KEY}': {content!r}")
delete_from_r2(KEY)